In [0]:
-- Processed Data
      -- 1 Batch Processing
      -- 2 Stream Processing

In [0]:
-- Incoming Raw (csv) Data into Delta (Batch / Processing)

In [0]:
-- create or replace table and build a medallion arch

In [0]:
-- Build a Medallion Arch for Batch / Streaming Data using Streaming Tables + AutoLoader (Cloud_Files)

In [0]:
CREATE OR REFRESH STREAMING TABLE yt_bronze 
AS
SELECT * FROM cloud_files("/Volumes/dea_databricks_catalog/default/dea_data/csv/", "csv", map("header","true","cloudFiles.inferColumnTypes","true"))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7744293879234361>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REFRESH STREAMING TABLE yt_bronze \nAS\nSELECT * FROM cloud_files("/Volumes/dea_demo/default/dea_volume1/yellowtaxi_raw/yellow/", "csv", map("header","true","cloudFiles.inferColumnTypes","true"))\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databri

In [0]:
-- Lets create a Silver Table too

In [0]:
CREATE OR REFRESH STREAMING TABLE yt_silver
(
  CONSTRAINT trip_distance_positive EXPECT (trip_distance > 0) ON VIOLATION DROP ROW,
  CONSTRAINT passenger_count_positive EXPECT (passenger_count > 0) ON VIOLATION DROP ROW,
  CONSTRAINT total_amount_positive EXPECT (total_amount > 0) ON VIOLATION DROP ROW,
  CONSTRAINT fare_amount_positive EXPECT (fare_amount > 0) ON VIOLATION DROP ROW,
  CONSTRAINT pick_up_drop_off_time_not_same EXPECT (tpep_pickup_datetime != tpep_dropoff_datetime) ON VIOLATION DROP ROW)
AS
SELECT * FROM STREAM(live.yt_bronze) 

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7744293879234363>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REFRESH STREAMING TABLE yt_silver\n(\n  CONSTRAINT trip_distance_positive EXPECT (trip_distance > 0) ON VIOLATION DROP ROW,\n  CONSTRAINT passenger_count_positive EXPECT (passenger_count > 0) ON VIOLATION DROP ROW,\n  CONSTRAINT total_amount_positive EXPECT (total_amount > 0) ON VIOLATION DROP ROW,\n  CONSTRAINT fare_amount_positive EXPECT (fare_amount > 0) ON VIOLATION DROP ROW,\n  CONSTRAINT pick_up_drop_off_time_not_same EXPECT (tpep_pickup_datetime != tpep_dropoff_datetime) ON VIOLATION DROP ROW)\nAS\nSELECT * FROM STREAM(live.yt_bronze) \n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   254

In [0]:
-- Lets create Gold Table/s

In [0]:
create or refresh streaming table yt_gold_1
as
SELECT VendorId, SUM(Total_amount) as Total_Revenue 
from
STREAM(live.yt_silver)
GROUP BY VendorId


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7744293879234365>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'create or refresh streaming table yt_gold_1\nas\nSELECT VendorId, SUM(Total_amount) as Total_Revenue \nfrom\nSTREAM(live.yt_silver)\nGROUP BY VendorId\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:1

In [0]:
create or refresh streaming table yt_gold_2
as
SELECT VendorId, SUM(trip_distance) as Total_Distance 
from
STREAM(live.yt_silver)
GROUP BY VendorId

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7744293879234366>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'create or refresh streaming table yt_gold_2\nas\nSELECT VendorId, SUM(trip_distance) as Total_Distance \nfrom\nSTREAM(live.yt_silver)\nGROUP BY VendorId\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py

In [0]:
-- End of the Notebook / Sessions